# 4. Monte Carlo uncertainty analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/04_monte_carlo_uncertainty.ipynb)

Prices, capital estimates, and process quantities are uncertain, and they are uncertain **at the same time**.
Monte Carlo simulation samples all of them from probability distributions, recomputes the plant thousands of
times, and returns distributions of NPV, levelized cost, and the other metrics instead of single numbers.

## Learning goals

- Define uncertainty distributions for prices, process quantities, and project-level factors.
- Run `monte_carlo()` and extract percentiles and probabilities such as P(NPV > 0).
- Check the sampled inputs and compare two plants under uncertainty.
- Link dependent parameters so that the analyses respect the process.

## 0) Setup: rebuild the green-ammonia plant

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia loop (from notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200,
                        process_type="Fluids",
                        category="Compressors, fans, & blowers",
                        type="Compressor, centrifugal", material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90,
                        process_type="Electrical",
                        category="Motors & generators",
                        type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250,
                        process_type="Fluids",
                        category="Heat exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000,
                        process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed",
                        material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25,
                        process_type="Fluids",
                        category="Pressure vessels", type="Vertical",
                        material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800,
                        process_type="Fluids",
                        category="Utilities",
                        type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20,
                         process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal",
                         material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller,
                  product_pump]

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    "process_type": "Fluids",
    "country": "Netherlands",
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,
    "project_lifetime": 20,      # years
    "plant_utilization": 0.90,   # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the first entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price
    "variable_opex_inputs": {
        # kg/day,  USD/kg
        "hydrogen":    {"consumption": 17_800, "price": 2.00},
        # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},
        # kWh/day, USD/kWh
        "electricity": {"consumption": 60_000, "price": 0.08},
    },
}

ammonia_plant = Plant(ammonia_config)
ammonia_plant.calculate_all()
print(f"Fixed capital investment : {ammonia_plant.fixed_capital:,.0f} USD")
print(f"Levelized cost of ammonia: {ammonia_plant.levelized_cost:.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:,.0f} USD")

In [ ]:
import numpy as np
from copy import deepcopy
from openpytea.analysis import *
from openpytea.plotting import *

## 1) Describing what you do not know

Uncertainty is configured **per item**, in the same configuration dictionary:

| Where | Key | Samples |
|----------------------------------|------------------------------|------------------------------------|
| `variable_opex_inputs` or `plant_products` | `price_uncertainty` | the price |
| `variable_opex_inputs` | `consumption_uncertainty` | the daily consumption |
| `plant_products` | `production_uncertainty` | the daily production |
| `operator_hourly_rate` | `rate_uncertainty` | the operator wage |
| top level | `project_uncertainties` | `fixed_capital_factor`, `fixed_opex_factor`, `project_lifetime`, `interest_rate`, `plant_utilization`, `tax_rate` |

Each uncertainty block describes one distribution. `dist_id` picks the family (normal by default), and the keys
it needs are:

| `dist_id` | Distribution | Keys |
|-----------|----------------------------|------------------------------------------|
| 2 | Lognormal | `loc`, `scale` (of ln *x*), `min`, `max` |
| 3 (default) | (truncated) Normal | `loc`, `scale`, `min`, `max` |
| 4 | Uniform | `min`, `max` |
| 5 | Triangular | `loc` (mode), `min`, `max` |

`loc` and `scale` have the more familiar aliases `mean` and `std`, so `{"std": 0.2}` is the same as
`{"scale": 0.2}`. If `loc` is left out it is the item's base value, and if `min`/`max` are left out a normal is
truncated at the base value ± 2 `std`. Bernoulli (6), discrete uniform (7), Weibull (8), gamma (9), beta (10), GEV (11), and Student's t
(12) are also available (see the [Analysis user guide](https://openpytea.readthedocs.io/en/stable/user_guide/analysis.html#configuring-input-uncertainties)).

The four project-level factors `fixed_capital_factor` (std 0.3), `fixed_opex_factor` (std 0.3),
`project_lifetime` (std 5 years), and `interest_rate` (std 0.03) are sampled **by default**; set `"std": 0` to
freeze one.

In [ ]:
ammonia_plant.update_configuration({
    "plant_products": {
        # triangular, mode at the base price 0.95
        "ammonia": {"price_uncertainty":
                    {"dist_id": 5, "min": 0.70, "max": 1.25}},
    },
    "variable_opex_inputs": {
        # truncated normal (the default)
        "hydrogen":    {"price_uncertainty":
                        {"std": 0.50, "min": 1.0, "max": 4.0}},
        "electricity": {"price_uncertainty":
                        {"std": 0.03, "min": 0.02, "max": 0.20}},
        "nitrogen":    {"price_uncertainty":
                        {"std": 0.01, "min": 0.02, "max": 0.10}},
    },
    "operator_hourly_rate": {
        "rate": 38.11,
        "rate_uncertainty": {"std": 8, "min": 25, "max": 60}},
    "project_uncertainties": {
        # +-30 %: the accuracy of a class-4 estimate
        "fixed_capital_factor": {"std": 0.30, "min": 0.6, "max": 1.6},
        "fixed_opex_factor":    {"std": 0.15, "min": 0.7, "max": 1.3},
        "interest_rate":        {"std": 0.015, "min": 0.04, "max": 0.12},
        "project_lifetime":     {"std": 0},  # frozen at 20 years
        # uniform, opt-in
        "plant_utilization":    {"dist_id": 4, "min": 0.80, "max": 0.95},
    },
})

## 2) Running the simulation

20 000 samples take a couple of seconds for this plant. Increase `num_samples` for smoother histograms;
`batch_size` controls memory use. `random_seed` makes the run reproducible.

In [ ]:
mc_base = monte_carlo(ammonia_plant, num_samples=20_000, batch_size=5_000,
                      random_seed=2026)

fig, ax = plot_monte_carlo(mc_base, metric="LCOP", bins=40,
                           label="Levelized cost of ammonia / [USD/kg]")
fig, ax = plot_monte_carlo(mc_base, metric="NPV", bins=40)

The result is a dictionary with the raw samples under `"metrics"` (`LCOP`, `NPV`, `ROI`, `PBT`) and the sampled
inputs under `"inputs"`. Percentiles and probabilities are one line of numpy:

In [ ]:
lcop = np.asarray(mc_base["metrics"]["LCOP"])
npv = np.asarray(mc_base["metrics"]["NPV"])

p10, p50, p90 = np.percentile(lcop, [10, 50, 90])
print(f"LCOP  P10 / P50 / P90 : {p10:.3f} / {p50:.3f} / {p90:.3f} USD/kg   "
      f"(deterministic: {ammonia_plant.levelized_cost:.3f})")
print(f"P(NPV > 0)            : {np.mean(npv > 0):.1%}")
print(f"P(LCOP < 0.95 USD/kg) : {np.mean(lcop < 0.95):.1%}")

## 3) Always look at the inputs

`plot_monte_carlo_inputs()` shows the sampled distributions, split into **process** quantities and **economic**
parameters. Inputs without an uncertainty block appear as a single bar at their fixed value, so you see what
varies and what does not.

In [ ]:
fig, axes = plot_monte_carlo_inputs(mc_base, category="economic", bins=40)

## 4) Comparing two plants under uncertainty

A deterministic comparison gives one number per design; Monte Carlo shows how often the ranking holds. Here we
compare the base case (local hydrogen at 2.00 USD/kg) with a plant that imports green hydrogen at about
3.50 USD/kg (± 0.25). All other uncertainties are shared, so only the hydrogen price sets them apart.

In [ ]:
imported = deepcopy(ammonia_plant)
imported.update_configuration({
    "plant_name": "Imported hydrogen",
    "variable_opex_inputs": {
        "hydrogen": {
            "price": 3.50,
            "price_uncertainty": {"std": 0.25, "min": 2.5, "max": 4.5}},
    },
})
imported.calculate_all()
mc_imported = monte_carlo(imported, num_samples=20_000, batch_size=5_000,
                          random_seed=2026)

fig, ax = plot_multiple_monte_carlo(
    [mc_base, mc_imported], metric="LCOP", bins=40,
    label="Levelized cost of ammonia / [USD/kg]")
print(f"Deterministic LCOP: {ammonia_plant.levelized_cost:.3f} "
      f"vs {imported.levelized_cost:.3f} USD/kg")
print(f"P(imported hydrogen is more expensive): "
      f"{np.mean(np.asarray(mc_imported['metrics']['LCOP']) > lcop):.1%}")

## 5) Going further: dependencies between parameters (short demo)

So far every input was sampled independently. In reality the feeds follow the ammonia production, and OpenPyTEA
lets you declare such links on the plant:

![Ammonia production drives the hydrogen, nitrogen, and electricity consumption](assets/dependency_graph.png)

Monte Carlo, `sensitivity_data()`, and `tornado_data()` all honour these links, and a dependent can no longer be
varied on its own.

In [ ]:
ammonia_plant.update_configuration({
    "plant_products": {
        # the root of the dependency graph
        "ammonia": {"production_uncertainty": {"std": 5_000, "min": 85_000,
                                               "max": 110_000}},
    },
    "variable_opex_inputs": {
        "hydrogen":    {"consumption_dependency": {
            "depends_on": {"production:ammonia": 17_800 / 100_000}}},
        "nitrogen":    {"consumption_dependency": {
            "depends_on": {"production:ammonia": 82_400 / 100_000}}},
        "electricity": {"consumption_dependency": {
            "depends_on": {"production:ammonia": 60_000 / 100_000}}},
    },
})

mc_linked = monte_carlo(ammonia_plant, num_samples=20_000, batch_size=5_000,
                        random_seed=2026)
fig, axes = plot_monte_carlo_inputs(mc_linked, category="process", bins=40)

In [ ]:
# The tornado now sweeps ammonia production together with the feeds it drives,
# and drops the dependents as separate bars
fig, ax = plot_tornado(tornado_data(ammonia_plant, plus_minus_value=0.3,
                                    metric="NPV", category="both"))

## 6) Task 4.1: how much would better cost data be worth?

The capital cost estimate is the largest single uncertainty (`fixed_capital_factor`, ± 30 %). Suppose a detailed
engineering study could narrow it to ± 10 %.

1. On a copy of `ammonia_plant`, set `fixed_capital_factor` to `{"std": 0.10, "min": 0.8, "max": 1.2}` and rerun
   Monte Carlo with the same seed.
2. Compare P10 / P90 of the NPV and P(NPV > 0) before and after. How much of the NPV spread was due to the capital estimate?
3. Bonus: repeat for the hydrogen price (`std` from 0.5 to 0.2). If you could pay to reduce only one of the two
   uncertainties, which would you pick?

In [ ]:
# YOUR CODE HERE

## Recap

- Uncertainty blocks sit next to the values they describe.
- `monte_carlo()` returns raw samples for percentiles and probabilities.
- Check the sampled inputs, and link dependent parameters to keep the sweeps physical.

**Next:** [5. Case study: steam methane reforming vs. water electrolysis](05_case_study_hydrogen.ipynb)